In [ ]:
# Ensure jacopy is importable when this notebook is opened
# directly (not via pytest). Walks up from the notebook's
# directory to the repo root and prepends it to sys.path if
# jacopy isn't already installed into this kernel.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401

# 19, The Courant family: Dorfman, Courant, Dirac

Companion notebook to [19_courant_family.md](19_courant_family.md). `CourantAlgebroid` carries both Courant and Dorfman brackets on shared Cartan operators; `DiracStructure` pins a maximally-isotropic involutive subbundle. The bridge identity, H-twisted Jacobi, and Dirac axioms all close as single citation-tagged proof steps.

## Section pairs

`SectionPair(vector, form)` wraps `(X, α) ∈ Γ(TM ⊕ T*M)`. Both brackets consume `SectionPair`s and produce one.

In [ ]:
from jacopy.brackets.dorfman import SectionPair
from jacopy.core.expr import Symbol
from jacopy.core.properties import Graded
from jacopy.core.registry import PropertyRegistry

reg = PropertyRegistry()
X = Symbol("X");  reg.declare(X, Graded(degree=0))
Y = Symbol("Y");  reg.declare(Y, Graded(degree=0))
alpha = Symbol("α"); reg.declare(alpha, Graded(degree=1))
beta  = Symbol("β"); reg.declare(beta,  Graded(degree=1))

a = SectionPair(X, alpha)
b = SectionPair(Y, beta)
print(f"a = ({a.vector}, {a.form})")
print(f"b = ({b.vector}, {b.form})")

## CourantAlgebroid, both brackets, shared operators

Construct with no arguments for untwisted; pass `background_H=H` to twist by a closed 3-form. Both brackets use the **same** Cartan operators, that sharing is what makes the bridge identity exact.

In [ ]:
from jacopy.library.courant_algebroid import CourantAlgebroid

C = CourantAlgebroid()
print(f"name           : {C.name}")
print(f"twisted        : {C.is_twisted}")

dorf = C.expand_dorfman(a, b, registry=reg)
cour = C.expand(a, b,         registry=reg)
print(f"Dorfman form half : {dorf.form}")
print(f"Courant form half : {cour.form}")

## The bridge identity

$$[a, b]_D - [a, b]_C = (0, \tfrac12\, d(\iota_X \beta + \iota_Y \alpha))$$

A single `theorem`-tagged step. Cartan's magic formula `L_Y α = d(ι_Y α) + ι_Y(dα)` makes the cancellation work.

In [ ]:
chain = C.prove_courant_dorfman_bridge(a, b, registry=reg)
step = chain.steps[0]
print(f"steps  : {len(chain)}")
print(f"rule   : {step.rule}")
print(f"tag    : {step.provenance_tag}")

correction = C.bridge_correction(a, b)
print(f"correction.vector : {correction.vector}")
print(f"correction.form   : {correction.form}")

## H-twisted Jacobi

Untwisted: Jacobi holds exactly (obstruction is `0`). H-twisted: Jacobi closes iff `dH = 0`, the obstruction lands on `dH`. Both produce single `axiom`-tagged steps.

In [ ]:
H = Symbol("H"); reg.declare(H, Graded(degree=3))
C_H = CourantAlgebroid(background_H=H)
print(f"name           : {C_H.name}")
print(f"twisted        : {C_H.is_twisted}")

chain = C_H.prove_jacobi_reduction(registry=reg)
step = chain.steps[0]
print(f"rule           : {step.rule}")
print(f"justification  : {step.justification}")

## DiracStructure, isotropy + involutivity

Pairing `⟨a, b⟩ = ½(ι_X β + ι_Y α)`; diagonal `⟨a, a⟩ = ι_X α` is the isotropy obstruction. Involutivity is `[a, b]_C ∈ Γ(L)`, surfaced as a placeholder symbol since subbundle membership isn't an Expr-level predicate.

In [ ]:
from jacopy.library.dirac import DiracStructure

L_sym = Symbol("L")
D     = DiracStructure(C, L_sym)
print(f"Dirac          : {D.name}")
print(f"⟨a, b⟩         : {D.pairing(a, b)}")
print(f"⟨a, a⟩         : {D.isotropy_obstruction(a)}")

chain_iso = D.prove_isotropy(a)
chain_inv = D.prove_involutivity(a, b)
print(f"isotropy   : {len(chain_iso)} step, rule={chain_iso.steps[0].rule}")
print(f"involutivity: {len(chain_inv)} step, rule={chain_inv.steps[0].rule}")

## The two canonical Dirac structures

| Factory | Subbundle | Source |
|---|---|---|
| `poisson_dirac(π)` | `L_π = {(π^♯ α, α)}` | Poisson bivector |
| `presymplectic_dirac(ω)` | `L_ω = {(X, ω^♭ X)}` | closed 2-form |

Each factory records the subbundle name; isotropy / involutivity remain axioms (the conditional theorems `dω = 0 ⇒ L_ω` Dirac, `[π,π] = 0 ⇒ L_π` Dirac are out of scope here).

In [ ]:
from jacopy.library.dirac import poisson_dirac, presymplectic_dirac

pi = Symbol("π"); reg.declare(pi, Graded(degree=1))
omega = Symbol("ω"); reg.declare(omega, Graded(degree=2))

L_pi    = poisson_dirac(pi,        courant=C)
L_omega = presymplectic_dirac(omega, courant=C)

print(f"poisson_dirac        : {L_pi.name}")
print(f"presymplectic_dirac  : {L_omega.name}")

## Summary

* `SectionPair(X, α)` operands for both brackets; `.vector` / `.form` accessors.
* `CourantAlgebroid` exposes `expand` (Courant) and `expand_dorfman` on shared Cartan operators.
* `prove_courant_dorfman_bridge`, single theorem-step asserting the exact correction `(0, ½ d(ι_X β + ι_Y α))`.
* `prove_jacobi_reduction`, vacuous when untwisted, lands obstruction on `dH` when H-twisted.
* `DiracStructure` carries pairing + isotropy + involutivity, all as axiom-tagged proof steps.
* `poisson_dirac` / `presymplectic_dirac` factories for the two source geometries.